a requesting data file

C:\Users\seifm\PycharmProjects\PythonProject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [20]:
from planner import NUM_WORKERS, BATCH_SIZE
%%writefile going_modular/data_getter.py
import os
from pathlib import Path
import zipfile
import requests

def data_getter (data_path ,
                 image_path,
                 link):
    """getting Data"""
    data_path_fn = Path(data_path)
    image_path_fn = data_path_fn / image_path
    full_zip_name = str(image_path_fn)+".zip"

    if image_path_fn.is_dir():
        print("The file there ,  downloading skipped")
    else:
        print("The file is downloading..... ")
        image_path_fn.mkdir(parents=True , exist_ok=True)
        with open(full_zip_name, 'wb') as f:
            request = requests.get(link)
            f.write(request.content)
        with zipfile.ZipFile(full_zip_name) as zip_ref:
            print("unzipping .....")
            zip_ref.extractall(image_path_fn)
            print("done")



Overwriting going_modular/data_getter.py


In [6]:
data_getter("../data", "food", "https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip")

The file there ,  downloading skipped


making a transform and imagefloder and data loader and get classes

In [21]:
%%writefile going_modular/data_setter_loader.py
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder

def data_setter_loader (train_dir,
                        test_dir,
                        transformer,
                        BATCH_SIZE,
                        NUM_WORKERS=os.cpu_count(),
                        ):
    """making a transform and imagefloder and Data loader and get classes"""
    #make the transform and the image folder
    train_data = ImageFolder(train_dir, transform=transformer)
    test_data = ImageFolder(test_dir, transform=transformer)
    #make the Data loader
    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    return train_loader, test_loader, train_data.classes







Writing going_modular/data_setter_loader.py


In [10]:

train_dir = Path("../data/food/train")
test_dir = Path("../data/food/test")

In [11]:
transformer = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])

In [35]:
data_setter_loader(train_dir,test_dir,transformer,32)

NameError: name 'data_setter_loader' is not defined

# making a tinyVGG

In [23]:
%%writefile going_modular/tinyVGG.py
from torch import nn
class VGGmodel (nn.Module):
    def __init__(self,
                 input_layers,
                 hidden_layers,
                 output_layers):
        super(VGGmodel, self).__init__()
        self.B1 = nn.Sequential(
            nn.Conv2d(in_channels = input_layers, out_channels = hidden_layers, kernel_size = 3,stride=1, padding = 0),
            nn.ReLU(),
            nn.Conv2d(in_channels = hidden_layers, out_channels = hidden_layers, kernel_size = 3,stride=1, padding = 0),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,
                     stride=2)
        )
        self.B2 = nn.Sequential(
            nn.Conv2d(in_channels = hidden_layers, out_channels = hidden_layers, kernel_size = 3,stride=1, padding = 0),
            nn.ReLU(),
            nn.Conv2d(in_channels = hidden_layers, out_channels = hidden_layers, kernel_size = 3,stride=1, padding = 0),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,
                     stride=2)
        )

        self.classifier =nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features = hidden_layers*13*13
                      , out_features = output_layers),
        )

    def forward (self,x):
        return self.classifier(self.B2(self.B1(x)))

Overwriting going_modular/tinyVGG.py


# making train and test steps

In [26]:
%%writefile going_modular/trainer.py
from tqdm.auto import tqdm

def train_step (model,
                dataloader_data,
                loss_fn,
                optimizer,
                device):
    model.train()
    train_loss , train_acc = 0,0
    for batch_idx, (data, label) in enumerate(dataloader_data):
        data, label = data.to(device), label.to(device)
        y_preds = model(data)
        loss = loss_fn(y_preds, label)
        train_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        #calculate the accuracy
        y_pred_class = torch.argmax(y_preds, dim=1)
        train_acc += (y_pred_class == label).sum().item()/len(y_preds)

    train_loss = train_loss / len(dataloader_data)
    train_acc = train_acc / len(dataloader_data)
    return train_loss, train_acc



def test_step (model
               ,dataloader_data,
                loss_fn,
                device):
    model.eval()
    test_loss,test_acc = 0,0
    with torch.inference_mode():
        for data, label in dataloader_data:
            data, label = data.to(device), label.to(device)
            y_preds = model(data)
            loss = loss_fn(y_preds, label)
            test_loss += loss.item()
            y_pred_class = torch.argmax(y_preds, dim=1)
            test_acc += (y_pred_class == label).sum().item()/len(y_preds)
    test_loss = test_loss / len(dataloader_data)
    test_acc = test_acc / len(dataloader_data)
    return test_loss, test_acc


def training_fn (model,
                dataloader_train_data,
                 dataloader_test_data,
                loss_fn,
                optimizer,
                EPOCHS,
                device):
    results={"train_loss":[]
            ,"train_acc":[]
            ,"test_loss":[],
            "test_acc":[]}
    for epoch in tqdm(range(EPOCHS)):
        train_loss,train_acc = train_step (model, dataloader_train_data, loss_fn, optimizer, device)
        test_loss , test_acc = test_step (model, dataloader_test_data, loss_fn, device)
        print(f"Epoch: {epoch} | Train loss: {train_loss:.4f} | Train acc: {train_acc*100:.4f} | Test loss: {test_loss:.4f} | Test acc: {test_acc*100:.4f}")
        results["train_loss"].append(train_loss)  #remember that the results is dictionary of lists that is why we make append
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)


    return results

Writing going_modular/trainer.py


# making a model saver

In [31]:
%%writefile going_modular/model_saver.py
def model_saver (model,
                 target_dir,
                 model_file_name):
    target_path = Path(target_dir)
    target_path.mkdir(parents=True,
                      exist_ok=True)
    assert model_file_name.endswith(".pth") or model_file_name.endswith(".pt")
    model_save_path  = target_path / model_file_name
    print(f"saving model to {model_save_path}")
    torch.save(model.state_dict(), model_save_path)

Writing going_modular/model_saver.py


# making the training thing

In [36]:
train_dir = "../data/food/train"
test_dir = "../data/food/test"

In [97]:

import torch
from torchvision import transforms
from torch import nn
from timeit import default_timer as timer
from finished_stuff.going_modular import data_setter_loader, trainer, tinyVGG

# MANUAL_SEED = 42
# torch.manual_seed(MANUAL_SEED)
# torch.cuda.manual_seed(MANUAL_SEED)
# np.random.seed(MANUAL_SEED)
# random.seed(MANUAL_SEED)

#setting hyperparameters
NUM_EPOCHES = 5
BATCH_SIZE = 32
HIDDEN_UNITS = 10
LR  = 0.0001
#make the device agnostic
device = "cuda" if torch.cuda.is_available() else "cpu"
#make transformer
transformer = transforms.Compose([transforms.Resize((64,64)),
                                  transforms.ToTensor(),])

In [98]:
#set Data
train_dataloader , test_dataloader , classes = data_setter_loader.data_setter_loader(train_dir, test_dir, transformer, BATCH_SIZE, 1)

In [99]:

#initiate the model
model = tinyVGG.VGGmodel(3, HIDDEN_UNITS, len(classes)).to(device)

In [100]:
#making the loss function and the optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

In [101]:
start_timer = timer()
model_results = trainer.training_fn(model, train_dataloader, test_dataloader, loss_fn, optimizer, NUM_EPOCHES, device)
end_timer = timer()
print(f"Training time: {end_timer - start_timer}")

 20%|██        | 1/5 [00:08<00:34,  8.52s/it]

Epoch: 0 | Train loss: 1.0975 | Train acc: 38.6719 | Test loss: 1.0986 | Test acc: 26.0417


 40%|████      | 2/5 [00:16<00:24,  8.29s/it]

Epoch: 1 | Train loss: 1.0997 | Train acc: 30.4688 | Test loss: 1.1030 | Test acc: 26.0417


 60%|██████    | 3/5 [00:25<00:16,  8.34s/it]

Epoch: 2 | Train loss: 1.0969 | Train acc: 30.4688 | Test loss: 1.1091 | Test acc: 26.0417


 80%|████████  | 4/5 [00:33<00:08,  8.23s/it]

Epoch: 3 | Train loss: 1.0936 | Train acc: 43.3594 | Test loss: 1.1147 | Test acc: 28.1250


100%|██████████| 5/5 [00:41<00:00,  8.24s/it]

Epoch: 4 | Train loss: 1.1041 | Train acc: 32.4219 | Test loss: 1.1177 | Test acc: 26.0417
Training time: 41.2052848000003
